In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from neo4j import GraphDatabase, basic_auth
import openai

driver = GraphDatabase.driver(
    "neo4j://52.4.166.125:7687",
    auth=basic_auth("neo4j", "stage-alkalinity-crashes")
)

In [3]:
cypher_query = '''
MATCH (m:Movie {title:$movie})<-[:RATED]-(u:User)-[:RATED]->(rec:Movie)
RETURN distinct rec.title AS recommendation LIMIT 20
'''

with driver.session(database='neo4j') as session:
    results = session.read_transaction(
        lambda tx: tx.run(cypher_query,
                          movie='Crimson Tide').data())
    for record in results:
        print(record['recommendation'])

<ipython-input-3-c1409c242bba>:7: DeprecationWarning: read_transaction has been renamed to execute_read
  results = session.read_transaction(


Mr. Holland's Opus
Apollo 13
Dead Man Walking
Seven (a.k.a. Se7en)
Heat
Get Shorty
Fugitive, The
Dave
Addams Family Values
True Lies
Speed
Lion King, The
Four Weddings and a Funeral
Forrest Gump
Star Trek: Generations
Shawshank Redemption, The
Stargate
Pulp Fiction
Outbreak
Miracle on 34th Street


In [3]:
from neo4j_genai.retrievers import Text2CypherRetriever
from neo4j_genai.llm import OpenAILLM

llm = OpenAILLM(model_name='gpt-4o', model_params={'temperature': 0})

In [4]:
from neo4j import GraphDatabase
from neo4j.time import Date

def get_node_datatype(value):
    '''
    입력된 노드 Value의 데이터 타입을 반환하는 함수
    '''
    if isinstance(value, str):
        return 'STRING'
    elif isinstance(value, int):
        return 'INTEGER'
    elif isinstance(value, float):
        return 'FLOAT'
    elif isinstance(value, bool):
        return 'BOOLEAN'
    elif isinstance(value, list):
        return f'LIST[{get_node_datatype(value[0])}]' if value else "LIST"
    elif isinstance(value, Date):
        return 'DATE'
    else:
        return 'UNKNOWN'

In [5]:
def get_schema(uri, user, password):
    '''
    Graph DB의 정보를 받아 노드 및 관계의 프로퍼티를 추출하고 스키마 딕셔너리를 반환하는 함수
    '''
    driver = GraphDatabase.driver(
        uri,
        auth=basic_auth(user, password)
    )

    with driver.session() as session:
        node_query = '''
        MATCH (n)
        WITH DISTINCT labels(n) AS node_labels, keys(n) AS property_keys, n
        UNWIND node_labels AS label
        UNWIND property_keys AS key
        RETURN label, key, n[key] AS sample_value
        '''
        nodes = session.run(node_query)

        rel_query = '''
        MATCH ()-[r]->()
        WITH DISTINCT type(r) AS rel_type, keys(r) AS property_keys, r
        UNWIND property_keys AS key
        RETURN rel_type, key, r[key] AS sample_value
        '''
        relationships = session.run(rel_query)
        
        rel_direction_query = '''
        MATCH (a)-[r]->(b)
        RETURN DISTINCT labels(a) AS start_label, type(r) AS rel_type, labels(b) AS end_label
        ORDER BY start_label, rel_type, end_label
        '''
        rel_directions = session.run(rel_direction_query)

        schema = {'nodes': {}, 'relationships': {}, 'relations': []}

        for record in nodes:
            label = record['label']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if label not in schema['nodes']:
                schema['nodes'][label] = {}
            schema['nodes'][label][key] = inferred_type
        
        for record in relationships:
            rel_type = record['rel_type']
            key = record['key']
            sample_value = record['sample_value']
            inferred_type = get_node_datatype(sample_value)
            if rel_type not in schema['relationships']:
                schema['relationships'][rel_type] = {}
            schema['relationships'][rel_type][key] = inferred_type
        
        for record in rel_directions:
            start_label = record['start_label'][0]
            rel_type = record['rel_type']
            end_label = record['end_label'][0]
            schema['relations'].append(f'(:{start_label})-[:{rel_type}]->(:{end_label})')
        
        return schema

def format_schema(schema):
    '''
        스키마 딕셔너리를 LLM에 제공하기 위해 원하는 형태로 formatting 하는 함수
    '''
    result = []

    result.append('Node properties:')
    for label, properties in schema['nodes'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{label} {{proprs}}')
    
    result.append('Relationship properties:')
    for rel_type, properties in schema['relationships'].items():
        props = ', '.join(f'{k}: {v}' for k, v in properties.items())
        result.append(f'{rel_type} {{{props}}}')
    
    result.append('The realtionships:')
    for relation in schema['relations']:
        result.append(relation)
    
    return '\n'.join(result)

In [6]:
schema = get_schema("neo4j://52.4.166.125:7687","neo4j", "stage-alkalinity-crashes")
neo4j_schema = format_schema(schema)
print(neo4j_schema)

Node properties:
Movie {proprs}
Genre {proprs}
User {proprs}
Actor {proprs}
Person {proprs}
Director {proprs}
Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}
DIRECTED {role: STRING}
The realtionships:
(:Actor)-[:ACTED_IN]->(:Movie)
(:Actor)-[:DIRECTED]->(:Movie)
(:Actor)-[:ACTED_IN]->(:Movie)
(:Director)-[:DIRECTED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:RATED]->(:Movie)


c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j\_sync\driver.py:542: ResourceWarning: unclosed  Neo4jDriver: <neo4j._sync.driver.Neo4jDriver object at 0x000001194D917110>.
  _unclosed_resource_warn(self)
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\neo4j\_sync\driver.py:547: DeprecationWarning: Relying on Driver's destructor to close the session is deprecated. Please make sure to close the session. Use it as a context (`with` statement) or make sure to call `.close()` explicitly. Future versions of the driver will not close drivers automatically.
  _deprecation_warn(


In [7]:
examples = [
    'USER INPUT: "Toy Story에 어떤 배우들이 출연하나요?" QUERY: MATCH (a:Actor)-[:ACTED_IN]->(m:Movie) WHERE m.title = "Toy Story" RETURN a.name',
    "USER INPUT: 'Toy Story의 평균 평점은 몇점인가요?' QUERY: MATCH (u:User)-[r:RATED]->(m:Movie) WHERE m.title = 'Toy Story' RETURN AVG(r.rating)",

    """
    USER INPUT: '저는 Toy Story 영화를 좋아합니다. Toy Story를 재밌게 본 사람은 또 어떤 영화를 재밌게 봤나요?'
    QUERY: MATCH (m:Movie)<-[r:RATED]-(u:User)-[recr:RATED]->(userBasedRec:Movie)
    WITH userBasedRec, COUNT(recr) AS recCount, AVG(recr.rating) AS avgRating
    ORDER BY avgRating DESC, recCount DESC
    RETURN DISTINCT userBasedRec.title, avgRating, recCount
    LIMIT 10
    """,

    """
    USER INPUT: '저는 'Wizard of Oz, The' 와 같은 영화를 좋아합니다. 이 영화와 비슷한 영화 추천해줄 수 있나요?'
    QUERY: MATCH (m:Movie) WHERE m.title = 'Wizard of Oz, The'
    MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
    WITH mm, rec, count(*) AS gs

    OPTIONAL MATCH (m)<-[:ACTED_IN]-(a)-[:ACTED_IN]->(rec)
    WITH m, rec, gs, count(a) AS as

    OPTIONAL MATCH (m)<-[:DIRECTED]-(d)-[:DIRECTED]->(rec)
    WITH m, rec, gs, as, count(d) AS ds

    RETURN rec.title AS recommendation,
            rec.poster AS rec_poster,
            gs AS genre_similarity,
            as AS actor_similarity,
            ds AS director_similarity,
            (5*gs)+(3*as)+(4*ds) AS score
    ORDER BY score DESC LIMIT 10
    """,

    """
    USER INPUT: '영화 Inception'과 비슷한 장르 혹은 비슷한 분위기의 영화를 추천해주세요.'
    QUERY: MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
    WHERE m.title = 'Inception' WITH rec, collect(g.name) AS genres, count(*) AS commonGenres
    RETURN rec.title, genres, commonGenres ORDER BY commonGenres DESC LIMIT 10;
    """
]

In [8]:
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples
)

query_text = 'Tom Hanks가 어떤 영화에 출연했나요?'
search_result = retriever.search(query_text=query_text)

In [10]:
search_result.items

[RetrieverResultItem(content="<Record m.title='Punchline'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Catch Me If You Can'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Dragnet'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Saving Mr. Banks'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Bachelor Party'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Volunteers'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Man with One Red Shoe, The'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Splash'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Big'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Nothing in Common'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Money Pit, The'>", metadata=None),
 RetrieverResultItem(content="<Record m.title='Toy Story of Terror'>", metadata=None),
 RetrieverResultItem(con

In [11]:
query_text = '저는 Titanic을 좋아합니다. 비슷한 영화를 추천해줄 수 있나요?'
search_result = retriever.search(query_text=query_text)

In [12]:
search_result.metadata['cypher']

"MATCH (m:Movie) WHERE m.title = 'Titanic'\nMATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)\nWITH m, rec, COUNT(g) AS genre_similarity\n\nOPTIONAL MATCH (m)<-[:ACTED_IN]-(a:Actor)-[:ACTED_IN]->(rec)\nWITH m, rec, genre_similarity, COUNT(a) AS actor_similarity\n\nOPTIONAL MATCH (m)<-[:DIRECTED]-(d:Director)-[:DIRECTED]->(rec)\nWITH rec, genre_similarity, actor_similarity, COUNT(d) AS director_similarity\n\nRETURN rec.title AS recommendation,\n       genre_similarity,\n       actor_similarity,\n       director_similarity,\n       (5*genre_similarity) + (3*actor_similarity) + (4*director_similarity) AS score\nORDER BY score DESC LIMIT 10"

In [13]:
print(search_result.metadata['cypher'])

MATCH (m:Movie) WHERE m.title = 'Titanic'
MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie)
WITH m, rec, COUNT(g) AS genre_similarity

OPTIONAL MATCH (m)<-[:ACTED_IN]-(a:Actor)-[:ACTED_IN]->(rec)
WITH m, rec, genre_similarity, COUNT(a) AS actor_similarity

OPTIONAL MATCH (m)<-[:DIRECTED]-(d:Director)-[:DIRECTED]->(rec)
WITH rec, genre_similarity, actor_similarity, COUNT(d) AS director_similarity

RETURN rec.title AS recommendation,
       genre_similarity,
       actor_similarity,
       director_similarity,
       (5*genre_similarity) + (3*actor_similarity) + (4*director_similarity) AS score
ORDER BY score DESC LIMIT 10


In [14]:
search_result.items

[RetrieverResultItem(content="<Record recommendation='Revolutionary Road' genre_similarity=2 actor_similarity=2 director_similarity=0 score=16>", metadata=None),
 RetrieverResultItem(content="<Record recommendation='Sense and Sensibility' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content="<Record recommendation='Reader, The' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content="<Record recommendation='Quills' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content='<Record recommendation="William Shakespeare\'s Romeo + Juliet" genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>', metadata=None),
 RetrieverResultItem(content="<Record recommendation='Total Eclipse' genre_similarity=2 actor_similarity=1 director_similarity=0 score=13>", metadata=None),
 RetrieverResultItem(content="

In [ ]:
from neo4j_genai.generation import GraphRAG

rag = GraphRAG(retriever=retriever, llm=llm)

: 

In [16]:
query_text = 'Titanic과 비슷한 장르의 영화 추천해주세용.'

response = rag.search(query_text=query_text, return_context=True)
print('==== [Text2Cypher 를 통해 자동생성한 Cypher] ====')
print(response.retriever_result.metadata['cypher'])
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cypher 를 통해 자동생성한 Cypher] ====
MATCH (m:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie) WHERE m.title = 'Titanic' WITH rec, collect(g.name) AS genres, count(*) AS commonGenres RETURN rec.title, genres, commonGenres ORDER BY commonGenres DESC LIMIT 10;

==== [생성된 Cypher를 기반으로 최종답변생성] ====
Here are some movies with similar genres to Titanic (Drama, Romance, Action):

1. Dirty Mary Crazy Larry
2. Shiri (Swiri)
3. Absolute Giganten
4. Eight Below
5. Kingdom of Heaven
6. Helen of Troy
7. Robin Hood
8. Legend of the Red Dragon (a.k.a. New Legend of Shaolin, The)
9. Casanova
10. House of Flying Daggers (Shi mian mai fu)


In [17]:
query_text = 'Toy Story와 The Godfather 영화를 좋아하는 사람은 또 어떤 영화를 좋아하나요?'

response = rag.search(query_text=query_text, return_context=True)
print('==== [Text2Cypher 를 통해 자동생성한 Cypher] ====')
print(response.retriever_result.metadata['cypher'])
print('\n==== [생성된 Cypher를 기반으로 최종답변생성] ====')
print(response.answer)

==== [Text2Cypher 를 통해 자동생성한 Cypher] ====
MATCH (m1:Movie)<-[r1:RATED]-(u:User)-[r2:RATED]->(otherMovies:Movie)
WHERE m1.title IN ['Toy Story', 'The Godfather']
WITH otherMovies, COUNT(r2) AS recCount, AVG(r2.rating) AS avgRating
ORDER BY avgRating DESC, recCount DESC
RETURN DISTINCT otherMovies.title, avgRating, recCount
LIMIT 10

==== [생성된 Cypher를 기반으로 최종답변생성] ====
Toy Story와 The Godfather 영화를 좋아하는 사람들은 다음과 같은 영화도 좋아할 수 있습니다:

1. Face in the Crowd, A
2. Paperman
3. Taste of Cherry (Ta'm e guilass)
4. Red Firecracker, Green Firecracker (Pao Da Shuang Deng)
5. Love & Human Remains
6. Lagaan: Once Upon a Time in India
7. Love Me If You Dare (Jeux d'enfants)
8. Village of the Damned
9. 'night Mother
10. The Guardian

이 영화들은 모두 평균 평점이 5.0으로 높은 평가를 받은 영화들입니다.


In [ ]:
import gradio as gr
from gradio.themes.base import Base

class Seafoam(Base):
    pass
seafoam = Seafoam()

with gr.Blocks(theme=seafoam) as demo:
    def default_llm(message):
        prompt_text = f'''
        당신은 영화 추천 시스템을 탑재한 챗봇입니다. user_input 에 대답하되, 사용자에게 좋아하는 영화나 장르 등을 말해보라고 권고해보세요.
        user_input : {message}
        '''
        return llm.invoke(prompt_text).content
    
    def intent_detection(message):
        prompt_text = f'''
        주어진 query_text가 영화 추천을 받기 위한 본격적인 질문으로 보이면 True를 반환하고, 그렇지 않으면 False를 반환해주세요.
        예시 : [('query_text': '안녕 반가워', 'answer': 'False'), ('query_text': '너가 영화추천을 그렇게 잘한다며?', 'answer': 'False'), ('query_text': 'Titanic 영화와 비슷한 장르의 영화를 추천해줄래?', 'answer': 'True')]
        user_input : {message}
        '''
        return llm.invoke(prompt_text).content == 'True'
    
    def response(message, chat_history):
        if(intent_detection(message)):
            rag_result = rag.search(query_text=message
                                    + "(Please also provide evidence for how you used context in your answer. YOU MUST ANSWER in KOREAN PLEASE.)"
                                    , return_context=True)
            chat_history.append((message, rag_result.answer))
            return chat_history, rag_result.retriever_result.metadata['cypher'], rag_result.retriever_result.items
        else:
            llm_result = default_llm(message)
            chat_history.append((message, llm_result))
            return chat_history, '영화 관련 질문이 아니었어요.', '영화 관련 질문이 아니었어요.'

    with gr.Row():
        with gr.Column(scale=4):
            gr.HTML('''<div style='text-align: center; max-width: 1000px; margin: 10px auto;'>
                <div>
                    <h1>Graph RAG 챗봇 !</h1>
                </div>
                <p style='margin-bottom: 10px; font-size:95%'>
                    💭 Graph DB의 영화 리뷰 데이터셋을 기반으로 답변합니다. 답변에 사용한 DB 조회 결과를 함께 확인해보세요. </a>
                </p>
            </div>''')
    
    with gr.Row():
        with gr.Column(scale = 1):
            generated_query = gr.Textbox(label='생성된 Cypher 쿼리')
            query_result = gr.Textbox(label='쿼리 조회 결과')
        with gr.Column(scale = 3):
            chatbot = gr.Chatbot()
            msg = gr.Textbox(placeholder='어떤 영화를 추천받고 싶으신가요? (원하는 장르나 재밌게 봤던 영화를 함께 말하면 도움이 됩니다.)', label='입력')
            examples = gr.Examples(
                examples=[
                    "저는 'Net, The'와 같은 영화를 좋아합니다. 이 영화와 비슷한 영화 추천해줄 수 있나요?",
                    "영화 'Inception'과 비슷한 장르 혹은 비슷한 분위기의 영화를 추천해주세요."
                ],
                inputs=[msg],
            )
            with gr.Row():
                gr.HTML('''<div style='text-align: center; max-width: 500px; margin: 0px auto;'>
                    <div>
                        <h1>    </h1>
                    </p>
                </div>''')
                gr.HTML('''<div style='text-align: center; max-width: 500px; margin: 0px auto;'>
                    <div>
                        <h1>    </h1>
                    </p>
                </div>''')
                btn = gr.Button('Submit', variant='primary')
                clear = gr.Button('Clear')

    btn.click(fn=response, inputs=[msg, chatbot], outputs=[chatbot, generated_query, query_result])
    msg.submit(response, [msg, chatbot], [chatbot, generated_query, query_result])

    clear.click(lambda: None, None, msg, queue=False)

demo.launch(debug=True, share=True)

c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\starlette\formparsers.py:12: PendingDeprecationWarning: Please use `import python_multipart` instead.
  import multipart
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\gradio\routes.py:1215: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\fastapi\applications.py:4495: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  return self.router.on_event(event_type)


Running on local URL:  http://127.0.0.1:7860


c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\gradio\analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.43.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Running on public URL: https://10c86991b8d34caab6.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


c:\Users\Jo\AppData\Local\Programs\Python311\Lib\site-packages\starlette\templating.py:176: DeprecationWarning: The `name` is not the first parameter anymore. The first parameter should be the `Request` instance.
Replace `TemplateResponse(name, {"request": request})` by `TemplateResponse(request, name)`.
  warnings.warn(
